In [1]:
import json 
import re
import random
from pathlib import Path
from datasets import load_dataset , concatenate_datasets
from transformers import AutoTokenizer
from datetime import date, timedelta


In [2]:
import os
os.chdir(os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "..")

In [ ]:
BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B-Instruct'


HERMES_DATASET = "NousResearch/hermes-function-calling-v1"

OUTPUT_DIR = Path('training/data')

TURKISH_DIR = Path('training/turkish_examples')

EVAL_RATIO = 0.04 

MAX_NEW_TOKENS = 4096

HERMES_SAMPLE_SIZE = 9_000

HERMES_CONFIGS = ["func_calling", "func_calling_singleturn", "glaive_func_calling"]

SEED = 42 

ROLE_MAP = {
    'system' : 'system',
    'user' : 'user' ,
    'human' : 'user' ,
    'assistant' : 'assistant' ,
    'gpt' : 'assistant' ,
    'tool' : 'tool' , 
    'observation' : 'tool'
}

CHATML_PATTERN = re.compile(r"<\|im_start\|>[ \t]*\w*|<\|im_end\|>")

In [4]:
DATE_RANGE_START = date(2024, 1, 1)
DATE_RANGE_END = date(2026, 12, 31)

# Ay kısaltmaları elle yazıldı: strftime("%b") işletim sisteminin diline bağlıdır
# ve Türkçe locale'de "Jul" yerine "Tem" üretir — bu, template formatını bozar.
_MONTHS_EN = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
            "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

In [5]:
def strip_chatml_markers(text : str ) -> str  : 
    
    cleaned = CHATML_PATTERN.sub('' , text)

    return cleaned.strip()

In [6]:
def normalize_messages(raw_turns : list[dict]) -> list[dict] | None : 

    if raw_turns is None : 
        return None 

    messages = []

    for turn in raw_turns : 
        raw_role = turn.get('from') or turn.get('role') or ''

        raw_role = raw_role.strip().lower()

        role = ROLE_MAP.get(raw_role)
        if raw_role is None : 
            return None

        raw_content = turn.get('value') or turn.get('content') or ''

        content = strip_chatml_markers(text = raw_content)

        if not content : 
            return None

        messages.append({'role' : role , 'content' : content})

    return messages
        

In [7]:
def random_date_string(rng: random.Random) -> str:

    span = (DATE_RANGE_END - DATE_RANGE_START).days

    d = DATE_RANGE_START + timedelta(days=rng.randint(0, span))
    
    return f"{d.day} {_MONTHS_EN[d.month - 1]} {d.year}"

In [8]:
def is_valid_conversation(messages : list[dict]) -> bool : 

    if len(messages) < 2 :
        return False 

    if messages[-1]['role'] != 'assistant' : 
        return False 

    if not any(m['role'] == 'assistant' for m in messages) :
        return False

    return True

In [9]:
def load_turkish_examples() -> list[list[dict]] :

    conversation =[]

    if not TURKISH_DIR.exists() : 
        return conversation

    for path in sorted(TURKISH_DIR.glob('*.jsonl')) : 

        with path.open('r' , encoding = 'utf-8') as f :

            for line_no , line in enumerate(f , start = 1 ) :

                line = line.strip()

                if not line : 
                    continue

                try : 
                    record = json.loads(line)
                except json.JSONDecodeError as exc :
                    raise ValueError(
                        f'Broken json : {path} line number {line_no} -> {exc}'
                    )

                messages = normalize_messages(record.get('messages' , []))

                if messages is None or not is_valid_conversation(messages) :
                    raise ValueError(
                        f'Invalid speech structure : {path} line no {line_no}'
                    )

                conversation.append(messages)

    return conversation


In [10]:
def verify_rendering(text: str, messages: list[dict]) -> None:

    if "<|start_header_id|>" not in text:
        raise RuntimeError(
            "Llama-3.1 header token not found. Chat template might not have been applied."
        )

    if "<|eot_id|>" not in text:
        raise RuntimeError("End-of-turn token (<|eot_id|>) not found.")

    if "<|start_header_id|>assistant<|end_header_id|>" not in text:
        raise RuntimeError("Assistant header not found; loss masking will not work.")

    if "<|im_start|>" in text or "<|im_end|>" in text:
        raise RuntimeError("ChatML tag remaining in output; sanitization failed.")

    if any(m["role"] == "tool" for m in messages):
        if "ipython" not in text and "tool" not in text:
            raise RuntimeError(
                "Tool turn present in conversation history, but no corresponding output found."
            )

In [11]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

per_config_dataset = []

for config_name in HERMES_CONFIGS : 
    per_config_dataset.append(
        load_dataset(HERMES_DATASET , config_name , split = 'train'
        )
    )

hermes = concatenate_datasets(per_config_dataset)

hermes = hermes.shuffle(seed = SEED)

if len(hermes) > HERMES_SAMPLE_SIZE : 
    hermes = hermes.select(range(HERMES_SAMPLE_SIZE))

In [12]:
len(hermes)

8995

In [13]:
hermes[100]

{'id': '2ef73e38-def6-4109-a3d9-a8034e73440b',
 'conversations': [{'from': 'system',
   'value': 'You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags.You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.Here are the available tools:<tools>\n[{"type": "function", "function": {"name": "get_random_number", "description": "Get a random number within a specified range", "parameters": {"type": "object", "properties": {"min": {"type": "number", "description": "The minimum value of the range"}, "max": {"type": "number", "description": "The maximum value of the range"}}, "required": ["min", "max"]}}}, {"type": "function", "function": {"name": "calculate_percentage", "description": "Calculate the percentage of a number", "parameters": {"type": "object", "properties": {"number": {"type": "number", "description": "The number to calculate the percentage of"}, "per

In [14]:
hermes_conversation = []
skipped = 0 

for row in hermes : 
    messages = normalize_messages(row.get('conversations' , []))

    if messages is None or not is_valid_conversation(messages) : 
        skipped += 1
        continue

    hermes_conversation.append(messages)

    if skipped > len(hermes) * 0.2 : 
        print('Check the data , high elimination rate')

In [15]:
len(hermes) == len(hermes_conversation) + skipped

True

In [16]:
hermes_conversation[1]

[{'role': 'system',
  'content': 'You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags.You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.Here are the available tools:<tools>\n[{"type": "function", "function": {"name": "search_movie", "description": "Search for information about a movie", "parameters": {"type": "object", "properties": {"title": {"type": "string", "description": "The title of the movie"}, "year": {"type": "integer", "description": "The release year of the movie"}, "genre": {"type": "string", "description": "The genre of the movie"}}, "required": ["title"]}}}, {"type": "function", "function": {"name": "create_invoice", "description": "Create a new invoice", "parameters": {"type": "object", "properties": {"customer_name": {"type": "string", "description": "The name of the customer"}, "items": {"type": "array", "items": {"type": "object"

In [17]:
turkish_conversation = load_turkish_examples()
if not turkish_conversation : 
    print('There are no turkish examples')




In [18]:
records = [] 
date_rng = random.Random(SEED)

for conversation , source in [
    (hermes_conversation , 'hermes') ,
    (turkish_conversation , 'turkish')
] :
    for messages in conversation :
        text = tokenizer.apply_chat_template(
            messages ,
            tokenize = False , 
            add_generation_prompt = False ,
            date_string=random_date_string(date_rng),
    )
        records.append({'text' : text , 'messages' : messages , 'source' : source})

In [19]:
kept = []
too_long = 0

for record in records : 
    n_tokens = len(tokenizer(record['text'] , add_special_tokens = False)['input_ids'])

    if n_tokens > MAX_NEW_TOKENS :
        too_long += 1
        continue

    kept.append(record)

    print(f'{too_long} data points were eliminated.')

print(f"Uzunluk filtresi: {len(kept)} kaldı, {too_long} elendi (>{MAX_NEW_TOKENS} token)")
records = kept

0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data points were eliminated.
0 data p

In [20]:
for source in ("hermes", "turkish"):

    sample = next((r for r in records if r["source"] == source), None)
    if sample is not None:
        verify_rendering(sample["text"], sample["messages"])

In [21]:
rng = random.Random(SEED)

rng.shuffle(records)

eval_size = int(len(records) * EVAL_RATIO)

eval_records = records[: eval_size]
train_records = records[eval_size: ]

turkish_in_eval = sum(1 for r in eval_records if r['source'] == 'turkish')

if turkish_conversation and turkish_in_eval < 10 :
    print('There are no enough turkish text in evaluation datas')




In [22]:
turkish_in_eval

23

In [23]:
OUTPUT_DIR.mkdir(parents = True , exist_ok = True)

for name , subset in [('train' , train_records) , ('eval' , eval_records)] :
    out_path = OUTPUT_DIR / f'{name}.jsonl'

    with out_path.open('w' , encoding = 'utf-8') as f :
        for record in subset :
            f.write(json.dumps(record , ensure_ascii = False ) + '\n')

    n_turkish = sum(1 for r in subset if r["source"] == "turkish")
    print(f"{out_path}: {len(subset)} example ({n_turkish} turkish)")

training/data/train.jsonl: 8781 example (477 turkish)
training/data/eval.jsonl: 365 example (23 turkish)
